In [3]:
import carla
import math
import random
import time
import numpy as np
import cv2

client = carla.Client('localhost', 2000)
client.reload_world()
world = client.get_world()

bp_lib = world.get_blueprint_library()
spawn_points = world.get_map().get_spawn_points()

In [4]:
vehicle_bp = bp_lib.find('vehicle.tesla.model3')
vehicle = world.try_spawn_actor(vehicle_bp, random.choice(spawn_points))
#vehicle.show_debug_telemetry(True)
camera_bp = bp_lib.find('sensor.camera.rgb')
camera_bp.set_attribute('image_size_x', '1600')
camera_bp.set_attribute('image_size_y', '900')
camera_bp.set_attribute('fov', '70')

# 6 camera suite setup
spectator = world.get_spectator()

camera_f_trans = carla.Transform(carla.Location(z=1.5))                            # front middle
camera_fl_trans = carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=-55))  # front left
camera_fr_trans = carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=+55))  # front right
camera_b_trans = carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=+180))  # back middle
camera_bl_trans = carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=-120)) # back left
camera_br_trans = carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=+120)) # back right

In [5]:
# FRONT camera spawn
f_camera = world.spawn_actor(camera_bp, camera_f_trans, attach_to=vehicle)

time.sleep(2)
spectator.set_transform(f_camera.get_transform())
#f_camera.destroy()

In [ ]:
# FRONT LEFT camera spawn
fl_camera = world.spawn_actor(camera_bp, camera_fl_trans, attach_to=vehicle)

time.sleep(2)
spectator.set_transform(fl_camera.get_transform())
fl_camera.destroy()

In [ ]:
# FRONT RIGHT camera spawn
fr_camera = world.spawn_actor(camera_bp, camera_fr_trans, attach_to=vehicle)

time.sleep(2)
spectator.set_transform(fr_camera.get_transform())
fr_camera.destroy()

In [ ]:
# BACK MIDDLE camera spawn
b_camera = world.spawn_actor(camera_bp, camera_b_trans, attach_to=vehicle)

time.sleep(2)
spectator.set_transform(b_camera.get_transform())
b_camera.destroy()

In [ ]:
# BACK LEFT camera spawn
bl_camera = world.spawn_actor(camera_bp, camera_bl_trans, attach_to=vehicle)

time.sleep(2)
spectator.set_transform(bl_camera.get_transform())
bl_camera.destroy()

In [29]:
# BACK RIGHT camera spawn
# remember, this camera has 110 fov

br_camera_bp = bp_lib.find('sensor.camera.rgb')
br_camera_bp.set_attribute('image_size_x', '1600')
br_camera_bp.set_attribute('image_size_y', '900')
br_camera_bp.set_attribute('fov', '110')
br_camera = world.spawn_actor(br_camera_bp, camera_br_trans, attach_to=vehicle)

time.sleep(2)
spectator.set_transform(br_camera.get_transform())
#br_camera.destroy()

In [6]:
import csv

# open the CSV once and keep the writer around; the callback writes each frame.
# re-running this cell starts a fresh file.
fieldnames = ['frame_id', 'timestamp', 'speed', 'input_throttle', 'input_steer',
              'input_brake', 'x', 'y', 'z', 'pitch', 'yaw', 'roll']

csv_file = open('output.csv', 'w', newline='')
csv_writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
csv_writer.writeheader()

def camera_callback(image, data_dict):
    data_dict['image'] = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))

    # query current vehicle state (this runs on CARLA's sensor thread)
    control = vehicle.get_control()
    transform = vehicle.get_transform()
    velocity = vehicle.get_velocity()

    loc = transform.location
    rot = transform.rotation
    speed = math.sqrt(velocity.x**2 + velocity.y**2 + velocity.z**2)  # m/s

    csv_writer.writerow({
        'frame_id': image.frame,
        'timestamp': image.timestamp,
        'speed': speed,
        'input_throttle': control.throttle,
        'input_steer': control.steer,
        'input_brake': control.brake,
        'x': loc.x,
        'y': loc.y,
        'z': loc.z,
        'pitch': rot.pitch,
        'yaw': rot.yaw,
        'roll': rot.roll,
    })
    csv_file.flush()  # crash-safe: each row hits disk. drop this for max callback speed.

In [7]:
image_w = camera_bp.get_attribute('image_size_x').as_int()
image_h = camera_bp.get_attribute('image_size_y').as_int()

camera_data = {'image': np.zeros((image_w, image_h, 4))}

# need to set the camera to the correct one, in the future need to all 6 at the same time
# doing front middle for now
f_camera.listen(lambda image: camera_callback(image, camera_data))

In [8]:
vehicle.set_autopilot(True)

In [9]:
window_name = 'RGB Camera'
cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE)
cv2.imshow(window_name, camera_data['image'])
cv2.waitKey(1)

while True:
    cv2.imshow(window_name, camera_data['image'])

    if cv2.waitKey(1) == ord('q'):
        break

cv2.destroyAllWindows()

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/tygo/code/vla_research/.venv/lib/python3.11/site-packages/cv2/qt/plugins"
QFontDatabase: Cannot find font directory /home/tygo/code/vla_research/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tygo/code/vla_research/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tygo/code/vla_research/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tygo/code/vla_research/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
N

In [10]:
# stop logging and close the file (run after stopping the loop above)
f_camera.stop()
csv_file.close()
print('telemetry written to output.csv')

telemetry written to output.csv
